# SHAP Analysis Notebook
This notebook runs the modular XAI pipeline and previews key results.

In [ ]:
from pathlib import Path
import pandas as pd

from src.data_loader import load_data
from src.preprocessing import preprocess_data
from src.model import train_model, evaluate_model
from src.shap_explainer import shap_explain, validate_shap
from src.lime_explainer import lime_explain
from src.evaluation import compare_methods

In [ ]:
RANDOM_STATE = 42
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = BASE_DIR / 'data' / 'dataset.csv'
RESULTS_DIR = BASE_DIR / 'results'
PLOTS_DIR = RESULTS_DIR / 'plots'

X, y = load_data(DATA_PATH)
prep = preprocess_data(X, y, test_size=0.2, random_state=RANDOM_STATE)
model = train_model(prep.X_train_transformed, prep.y_train, random_state=RANDOM_STATE)
metrics = evaluate_model(model, prep.X_test_transformed, prep.y_test)
print('Accuracy:', metrics['accuracy'])
print('ROC-AUC:', metrics['roc_auc'])

In [ ]:
instance_index = 0
shap_artifacts = shap_explain(model, prep.X_train_transformed, prep.X_test_transformed, PLOTS_DIR, instance_index)
validation = validate_shap(model, prep.X_test_transformed, shap_artifacts)
lime_df = lime_explain(model, prep.X_train_transformed, prep.X_test_transformed, PLOTS_DIR, instance_index, RANDOM_STATE)
comparison_df = compare_methods(shap_artifacts.shap_values, prep.feature_names, lime_df, model, PLOTS_DIR)
comparison_df.head(10)

In [ ]:
print('Model prediction:', validation['model_prediction'])
print('SHAP reconstruction:', validation['shap_reconstruction'])
print('Absolute difference:', validation['absolute_difference'])
print('Top LIME features:')
print(lime_df.head(10))